# 04 - Fraud Detection: three detectors

Build and compare the three detection layers the system combines:

1. **Supervised ML** - Logistic Regression, Random Forest, XGBoost
   (`src/detection/fraud_model.py`)
2. **Rule engine** - expert rules, no training needed (`src/detection/rule_based.py`)
3. **Anomaly detection** - Isolation Forest, trained *without* labels
   (`src/detection/anomaly_detection.py`)

> This notebook is self-contained: it trains its **own** models on a random
> sample and does **not** overwrite the production artefacts in `models/`.

**Prerequisite:** `python main.py` (see `01_data_exploration.ipynb`).

In [ ]:
import sys
from pathlib import Path

# Make 'import src...' work no matter where the kernel was started from.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_transactions.csv"
FEATURED_PATH = PROJECT_ROOT / "data" / "processed" / "featured_transactions.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "fraud_model.pkl"

print("Project root:", PROJECT_ROOT)

## 1. Sample and split

In [ ]:
from sklearn.model_selection import train_test_split
from src.detection.fraud_model import (
    sample_processed_data, engineer_features, get_feature_columns,
    train_model, evaluate_model,
)

# Streams the full cleaned CSV and keeps a random slice (a minute or so).
df = sample_processed_data(max_rows=60_000, random_state=42)
TARGET = "isfraud"

X, y = df.drop(columns=[TARGET]), df[TARGET]
idx_train, idx_test = train_test_split(
    X.index, test_size=0.25, random_state=42, stratify=y
)
print(f"train={len(idx_train)}  test={len(idx_test)}  "
      f"fraud in test={int(y.loc[idx_test].sum())}")

# Same protocol as the main pipeline: the large-amount threshold is fitted on
# TRAIN rows only, then reused for every split.
threshold = X.loc[idx_train, "amount"].median()
eng = engineer_features(pd.concat([X, y], axis=1), large_amount_threshold=threshold)
feature_cols = get_feature_columns(eng)
print(f"{len(feature_cols)} model features")

## 2. Supervised models

In [ ]:
results, models = {}, {}
for model_type in ["logistic_regression", "random_forest", "xgboost"]:
    print(f"--- {model_type} ---")
    bundle = train_model(eng.loc[idx_train, feature_cols], y.loc[idx_train],
                         model_type=model_type)
    models[model_type] = bundle
    results[model_type] = evaluate_model(
        bundle, eng.loc[idx_test, feature_cols], y.loc[idx_test]
    )

comparison = pd.DataFrame(results).T
comparison[["accuracy", "precision", "recall", "f1", "roc_auc"]].round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, res) in zip(axes, results.items()):
    cm = np.array(res["confusion_matrix"])
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{int(v):,}", ha="center", va="center",
                color="black" if v < cm.max() * 0.6 else "white")
    ax.set_xticks([0, 1], ["legit", "fraud"])
    ax.set_yticks([0, 1], ["legit", "fraud"])
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
    ax.set_title(name)
fig.tight_layout()
plt.show()

## 3. Rule engine

No training, no labels - just explicit, explainable conditions. On the same
held-out rows, how far do the rules get us?

In [ ]:
from collections import Counter

from src.detection.rule_based import apply_fraud_rules

y_test = y.loc[idx_test]
rule_out = apply_fraud_rules(X.loc[idx_test])
pred = rule_out["rule_based_flag"].astype(int)

tp = int(((pred == 1) & (y_test == 1)).sum())
fp = int(((pred == 1) & (y_test == 0)).sum())
fn = int(((pred == 0) & (y_test == 1)).sum())
rule_metrics = pd.Series({
    "precision": tp / max(1, tp + fp),
    "recall": tp / max(1, tp + fn),
    "flagged_share_%": 100 * float(pred.mean()),
}, name="rule engine").to_frame().T
display(rule_metrics)

print("Reason frequency across the test set:")
for reason, n in Counter(r for reasons in rule_out["rule_reasons"] for r in reasons).most_common():
    print(f"  {n:>6}  {reason}")

## 4. Anomaly detection (unsupervised)

Isolation Forest is trained on the **training rows without using the label**,
then asked to flag outliers in the test set. It can catch *novel* patterns the
rules and the supervised model have never seen - at the cost of precision.

In [ ]:
from src.detection.anomaly_detection import train_anomaly_detector, detect_anomalies

anom_bundle = train_anomaly_detector(df.loc[idx_train], contamination=0.005)
anom = detect_anomalies(df.loc[idx_test], anom_bundle)
pred_a = anom["is_anomaly"].values

tp = int(((pred_a == 1) & (y_test == 1)).sum())
fp = int(((pred_a == 1) & (y_test == 0)).sum())
fn = int(((pred_a == 0) & (y_test == 1)).sum())
pd.Series({
    "precision": tp / max(1, tp + fp),
    "recall": tp / max(1, tp + fn),
    "flagged_share_%": 100 * float(pred_a.mean()),
}, name="isolation forest").to_frame().T

## 5. Putting them together - the production scorer

`src/utils/risk_scoring.py` combines the three detectors into one explainable
score (`0.5 * ML + 0.3 * rules + 0.2 * anomaly`). It runs the **saved**
production models from `models/`, so this cell needs `python main.py` to have
been executed at least once.

In [ ]:
from src.utils.risk_scoring import combine_risk_scores, explain_risk

if MODEL_PATH.exists():
    demo = pd.concat([df[df[TARGET] == 1].head(2), df[df[TARGET] == 0].head(2)])
    scored = combine_risk_scores(demo.drop(columns=[TARGET]))
    display_cols = ["type", "amount", "fraud_probability", "rules_triggered",
                    "is_anomaly", "risk_score", "risk_level"]
    display(scored[display_cols])
    print("Explanation of the first row:")
    print(explain_risk(scored.iloc[0]))
else:
    print("models/ is empty - run 'python main.py' first to build the "
          "production models this cell needs.")

## Summary

| Detector | Needs labels? | Strength | Weakness |
|---|---|---|---|
| Supervised (RF/XGB) | yes | highest recall + ranking (ROC-AUC) | blind to new fraud *types*; precision suffers on a 0.1% base rate |
| Rules | no | instant, explainable, deterministic | fixed thresholds; easy to evade |
| Isolation Forest | no | catches never-seen patterns | noisy: most anomalies are just rare-but-legit |

- The supervised models win on recall/ROC-AUC; Logistic Regression acts as a
  linear baseline, Random Forest/XGBoost capture the non-linear balance rules.
- Rules alone already catch a large share of fraud with perfect
  explainability - they are the "cheap layer" that needs no retraining.
- The anomaly layer contributes a complementary *novelty* signal.
- **No single detector is sufficient** - which is exactly what the weighted
  risk score in `05_model_evaluation.ipynb` operationalises.